# 01 · Ticket routing: your first decision model

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/daseinlabs/open-jev/blob/main/cookbooks/01-ticket-routing/cookbook.ipynb)


Turn a CSV of labelled support tickets into a model that picks a **team** and a **priority** for
new tickets, says how confident it is, and hands off the tickets it isn't sure about.

You'll touch every part of the SDK once: a `DecisionSpec`, a dataset, training, calibration, and
confident-or-defer serving. About 10–20 minutes, mostly Gemma reading the tickets.

```
tickets.csv ──spec──▶ decision dataset ──train──▶ calibrated model ──decide──▶ team / priority / defer
```


**Runs on** an NVIDIA/AMD GPU with about 9 GB of memory (PyTorch) or an Apple silicon Mac (MLX); the
backend is picked for you. On Colab, choose an L4 or A100 runtime: a T4 has no bfloat16, and
Gemma in float32 needs about 17 GB. The backbone (Gemma 3 4B, 8 GB) downloads once from
[`parsecai/gemma-3-4b-it`](https://huggingface.co/parsecai/gemma-3-4b-it). Accept the Gemma terms on that page first.

In [1]:
# Setup: install the SDK if it's missing, log in to Hugging Face once (the backbone is gated).
import importlib.util
if importlib.util.find_spec('openjev') is None:
    %pip install -q openjev-sdk

from huggingface_hub import get_token, login
if get_token() is None:
    login()

import json, shutil
from pathlib import Path
from openjev import compute

def fresh(path):
    """Remove a previous run's output, so every cell can be re-run."""
    shutil.rmtree(path, ignore_errors=True)
    return Path(path)

print('backend:', compute.backend(), '| device:', 'metal' if compute.backend() == 'mlx' else compute.device())

backend: mlx | device: metal


## 0. The data

A sample of 1,200 labelled tickets. 30% mention two topics, and 5% are deliberately mislabelled,
as real routing logs are. **For your own data, export the same columns and skip this cell.**

In [2]:
import csv, random

TEMPLATES = {
    'billing': ["I was charged twice for {thing}.", "My invoice for {thing} shows the wrong amount.",
                "Can I get a refund on the {thing} subscription fee?", "Why did my card get billed again this month?"],
    'shipping': ["My {thing} still hasn't arrived.", "The tracking for my {thing} hasn't moved in a week.",
                 "Can you ship the {thing} to a different address?", "The courier left my {thing} at the wrong house."],
    'returns': ["The {thing} arrived broken, I want a replacement.", "How do I send back the {thing}?",
                "The {thing} is the wrong size, can I exchange it?", "I'd like to return the {thing} I bought last week."],
    'account': ["I can't log in to my account.", "Please change the email on my account.",
                "I want to delete my account and data.", "My password reset link isn't working."],
    'technical': ["The app crashes when I open my {thing} order.", "The website shows an error at checkout.",
                  "Your app won't load my order history.", "The {thing} setup guide link is broken."],
}
THINGS = ['headphones', 'blender', 'yoga mat', 'monitor', 'jacket', 'coffee maker', 'backpack', 'desk']
URGENT = ["This is urgent.", "I need this fixed today.", "Please help ASAP!", "This is the third time I'm asking."]
LOW = ["No rush.", "Whenever you get a chance.", "Just curious.", "Not a big deal."]

rng = random.Random(0)
with open('tickets.csv', 'w', newline='') as stream:
    writer = csv.writer(stream)
    writer.writerow(['ticket_id', 'customer_id', 'message', 'team', 'priority'])
    for i in range(1200):
        team = rng.choice(list(TEMPLATES))
        priority = rng.choices(['urgent', 'normal', 'low'], [1, 2, 1])[0]
        message = rng.choice(TEMPLATES[team]).format(thing=rng.choice(THINGS))
        if rng.random() < .3:  # ambiguous tickets mention a second topic
            other = rng.choice([t for t in TEMPLATES if t != team])
            message += ' Also, ' + rng.choice(TEMPLATES[other]).format(thing=rng.choice(THINGS)).lower()
        if rng.random() < .6:
            message += f' Order #{rng.randint(10000, 99999)}.'
        if priority != 'normal':
            message += ' ' + rng.choice(URGENT if priority == 'urgent' else LOW)
        if rng.random() < .05:  # real labels are noisy: some tickets were routed by mistake
            team = rng.choice(list(TEMPLATES))
        writer.writerow([f't{i:05d}', f'c{rng.randrange(400):04d}', message, team, priority])

tickets = list(csv.DictReader(open('tickets.csv')))
for t in tickets[:3]:
    print(t)

{'ticket_id': 't00000', 'customer_id': 'c0111', 'message': "My password reset link isn't working. Also, the jacket setup guide link is broken. Just curious.", 'team': 'account', 'priority': 'low'}
{'ticket_id': 't00001', 'customer_id': 'c0373', 'message': 'The website shows an error at checkout. Order #79804. I need this fixed today.', 'team': 'technical', 'priority': 'urgent'}
{'ticket_id': 't00002', 'customer_id': 'c0244', 'message': 'Can I get a refund on the desk subscription fee? Order #51444. Whenever you get a chance.', 'team': 'billing', 'priority': 'low'}


## 1. Describe the decisions

A **kind** is one type of decision: a question, plus a function that lists the candidates for a
state. Here both candidate lists are fixed.

- Candidate **text** is what the model reads, so describe each option
  (`'Returns: broken items, exchanges, send-backs'`, not just `returns`).
- The **payload** is what your code gets back.
- Bump a kind's `version` whenever its candidate list changes. The SDK then refuses to serve an
  old model with the new spec.

In [3]:
from openjev.tod import Candidate, DecisionSpec, Episode, Kind, build_dataset

TEAMS = {'billing': 'Billing: charges, invoices, refunds of fees',
         'shipping': 'Shipping: delivery, tracking, addresses',
         'returns': 'Returns: broken items, exchanges, send-backs',
         'account': 'Account: login, email, deletion',
         'technical': 'Technical: app or website errors'}
PRIORITIES = {'urgent': 'Urgent: handle today', 'normal': 'Normal', 'low': 'Low: no rush'}

def options(table):
    return lambda state: [Candidate(id, text, payload=id) for id, text in table.items()]

SPEC = DecisionSpec('ticket-routing', [
    Kind('team', 'Which team should handle this ticket?', options(TEAMS), version='1'),
    Kind('priority', 'How urgent is this ticket?', options(PRIORITIES), version='1'),
])

## 2. Build the dataset

Every row goes through the spec, so a model can later check that its data and its serving code
agree. The **state** is any JSON; everything in it is shown to the model. `group_id` keeps all
of a customer's tickets in one split, so test accuracy isn't inflated by near-duplicates the model saw in training.

The report shows rows per split and kind. Identical tickets are kept once (`duplicates_dropped`),
so the same text can't sit in both train and test.

In [4]:
episodes = []
for row in tickets:
    state = {'message': row['message']}
    episodes.append(Episode(row['ticket_id'], (
        SPEC.decision(f"{row['ticket_id']}:team", 'team', state, choice=row['team'], group_id=row['customer_id']),
        SPEC.decision(f"{row['ticket_id']}:priority", 'priority', state, choice=row['priority'], group_id=row['customer_id']),
    )))

dataset = build_dataset(SPEC, episodes)
DATASET = fresh('runs/tickets')
dataset.write(DATASET)
print(json.dumps({k: dataset.report[k] for k in ('splits', 'duplicates_dropped', 'duplicates_with_different_label')}, indent=1))

{
 "splits": {
  "train": {
   "priority": {
    "Choice": 717
   },
   "team": {
    "Choice": 717
   }
  },
  "validation": {
   "priority": {
    "Choice": 139
   },
   "team": {
    "Choice": 139
   }
  },
  "calibration": {
   "priority": {
    "Choice": 114
   },
   "team": {
    "Choice": 114
   }
  },
  "test": {
   "priority": {
    "Choice": 101
   },
   "team": {
    "Choice": 101
   }
  }
 },
 "duplicates_dropped": 258,
 "duplicates_with_different_label": 6
}


## 3. Train

Each kind gets its own small head (~2M parameters) on the shared, frozen backbone. Per kind:

1. **Features.** Gemma reads every ticket and candidate once; the hidden states are cached. This is most of the runtime.
2. **Head.** A cross-attention head learns to score candidates against the ticket; the best validation checkpoint is kept.
3. **Calibration.** A temperature is fitted on its own split, so "0.9 confident" means right about 90% of the time.
4. **Test.** Accuracy and mean confidence on tickets never used above.

The same step from a terminal: `openjev tod train runs/tickets --out runs/tickets-model`.

In [5]:
from openjev.tod.train import train_dataset

MODEL = fresh('runs/tickets-model')
result = train_dataset(DATASET, None, MODEL)  # None: $OPENJEV_MODEL, else parsecai/gemma-3-4b-it

priority: features (train)


Extracted 100/717


Extracted 200/717


Extracted 300/717


Extracted 400/717


...

{"epoch": 17, "train_loss": 0.1775419602749903, "val_top1": 0.935251798561151, "val_top3": 0.9640287769784173, "val_ece": 0.045171957077228875, "val_nll": 0.43382430725615817, "val_brier": 0.12787531052904086, "seconds": 0.1}


{"epoch": 18, "train_loss": 0.17240252263708927, "val_top1": 0.9280575539568345, "val_top3": 0.9712230215827338, "val_ece": 0.05344352351536307, "val_nll": 0.49729843995212086, "val_brier": 0.13944146385183612, "seconds": 0.1}


{"epoch": 19, "train_loss": 0.1832009435552101, "val_top1": 0.920863309352518, "val_top3": 0.9640287769784173, "val_ece": 0.07026146947502517, "val_nll": 0.5441402258664284, "val_brier": 0.15355990651250992, "seconds": 0.1}


{"epoch": 20, "train_loss": 0.1663481712242591, "val_top1": 0.920863309352518, "val_top3": 0.9712230215827338, "val_ece": 0.04760997786154989, "val_nll": 0.5175772709968285, "val_brier": 0.15061077225677014, "seconds": 0.1}


calibrating and evaluating


In [6]:
for kind, r in result['test'].items():
    print(f"{kind:9} accuracy {r['accuracy']:.1%}   mean confidence {r['mean_confidence']:.2f}   n={r['n']}")

priority  accuracy 100.0%   mean confidence 1.00   n=101
team      accuracy 97.0%   mean confidence 0.90   n=101


Team routing is the harder kind: some tickets mention two topics, and 5% of labels are wrong
(test included). Mean confidence close to accuracy means the probabilities are honest.

## 4. Serve: act when confident, defer otherwise

In [7]:
from openjev.tod.rl import HeadPolicy

policy = HeadPolicy(MODEL, SPEC)  # refuses a model trained with a different spec

for message in ['My parcel never showed up and I need it for tomorrow!',
                'I was charged twice and also my parcel is late',
                'hello?']:
    print(message)
    for kind in SPEC.kinds:
        p = policy.decide({'message': message}, kind, threshold=0.8)
        print(f'  {kind:9} -> {p.choice_id:10} confidence {p.confidence:.2f}' + ('   DEFER' if p.defer else ''))

My parcel never showed up and I need it for tomorrow!
  team      -> shipping   confidence 0.96


  priority  -> urgent     confidence 1.00
I was charged twice and also my parcel is late
  team      -> billing    confidence 1.00
  priority  -> normal     confidence 1.00
hello?


  team      -> billing    confidence 0.40   DEFER
  priority  -> normal     confidence 1.00


In your code:

```python
p = policy.decide(state, 'team', threshold=0.8)
if p.defer:
    ...                 # hand the ticket to a person or a bigger model
else:
    route_to(p.payload)
```

## 5. Pick the threshold

The threshold is the product dial: raise it, and the model makes fewer mistakes but hands off more.
Here is the trade-off on the test tickets:

In [8]:
from openjev.tod.spec import load_decisions

test = [d for d in load_decisions(DATASET/'test.jsonl') if d.kind == 'team']
predictions = [(policy.decide(d.state, 'team'), d.target.choice_id) for d in test]
print('threshold  handled  accuracy on handled')
for threshold in (0.0, 0.5, 0.7, 0.8, 0.9, 0.95):
    kept = [(p, y) for p, y in predictions if p.confidence >= threshold]
    accuracy = sum(p.choice_id == y for p, y in kept) / max(1, len(kept))
    print(f'{threshold:9.2f}  {len(kept) / len(predictions):7.0%}  {accuracy:19.1%}')

threshold  handled  accuracy on handled
     0.00     100%                97.0%
     0.50     100%                97.0%
     0.70      89%                96.7%
     0.80      83%                96.4%
     0.90      73%                97.3%
     0.95      57%                96.6%


## Your data

- **Columns:** change the loop in step 2. The state can hold anything (subject, body, customer tier, ...).
- **Candidates:** edit `TEAMS` / `PRIORITIES`, and bump that kind's `version`.
- **More kinds:** add a `Kind` and one more `SPEC.decision(...)` per record.

Next: [02 · traces → SFT → RL](../02-shopdesk-traces-to-rl/cookbook.ipynb).